# REST API: Alpha Vantage Tesla News

In this notebook, Tesla-related financial news are collected from the Alpha Vantage News & Sentiment REST API.  
The API data is cleaned and saved as structured CSV files for later merging with Tesla stock data and scraped news data.

# Imports

In [1]:
import ast
import os
from pathlib import Path

import pandas as pd
import requests

# API Request

In [2]:
RAW_PATH = Path("../data/raw/alpha_vantage_tesla_news_raw.csv")
PROCESSED_PATH = Path("../data/processed/alpha_vantage_tesla_news_cleaned.csv")

# For a public repository, the API key should not be hardcoded.
# Set ALPHA_VANTAGE_API_KEY in the environment when rerunning the notebook.
api_key = os.getenv("ALPHA_VANTAGE_API_KEY", "demo")

url = (
    "https://www.alphavantage.co/query"
    "?function=NEWS_SENTIMENT"
    "&tickers=TSLA"
    "&time_from=20260601T0000"
    "&time_to=20260702T0000"  
    "&sort=EARLIEST" 
    "&limit=1000"
    f"&apikey={api_key}"
)

try:
    response = requests.get(url, timeout=30)
    print("Status code:", response.status_code)
    data = response.json()
except Exception:
    data = {}

if "feed" in data and data["feed"]:
    articles = data["feed"]
    api_raw_df = pd.DataFrame(articles)
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    api_raw_df.to_csv(RAW_PATH, index=False)
    print("Loaded fresh API data and updated raw cache.")
elif RAW_PATH.exists():
    api_raw_df = pd.read_csv(RAW_PATH)
    articles = api_raw_df.to_dict(orient="records")
    print("No fresh API feed available in this environment. Loaded cached raw API data from file.")
else:
    raise RuntimeError("No API feed was returned and no cached raw API file exists.")

print("Number of API rows:", len(api_raw_df))


Status code: 200
No fresh API feed available in this environment. Loaded cached raw API data from file.
Number of API rows: 50


# Check API Response

In [3]:
print("Raw API columns:")
print(api_raw_df.columns)

display(api_raw_df.head())

Raw API columns:
Index(['title', 'url', 'time_published', 'authors', 'summary', 'banner_image',
       'source', 'category_within_source', 'source_domain', 'topics',
       'overall_sentiment_score', 'overall_sentiment_label',
       'ticker_sentiment'],
      dtype='object')


,title,url,time_published,authors,summary,banner_image,source,category_within_source,source_domain,topics,overall_sentiment_score,overall_sentiment_label,ticker_sentiment
0,UBS reiterates Packaging Corp. of America stoc...,https://ca.investing.com/news/stock-market-new...,20260611T143829,['Investing.com'],UBS has reiterated a Buy rating and a $248.00 ...,https://i-invdn-com.investing.com/news/LYNXMPE...,Investing.com Canada,General,Investing.com Canada,"[{'topic': 'earnings', 'relevance_score': '1.0...",0.049732,Neutral,"[{'ticker': 'PKG', 'relevance_score': '1.00000..."
1,Oppenheimer raises Tesla stock price target on...,https://www.investing.com/news/analyst-ratings...,20260611T132540,['Investing.com'],Oppenheimer has increased its price target for...,https://i-invdn-com.investing.com/news/moved_L...,Investing.com,General,Investing.com,"[{'topic': 'financial_markets', 'relevance_sco...",0.203912,Somewhat-Bullish,"[{'ticker': 'TSLA', 'relevance_score': '1.0000..."
2,Is the 2026 Toyota bZ Woodland a Good Crossove...,https://www.cars.com/articles/is-the-2026-toyo...,20260611T113909,['Jim Travers'],The 2026 Toyota bZ Woodland is an electric cro...,https://images.cars.com/cldstatic/wp-content/u...,Cars.com,General,Cars.com,"[{'topic': 'energy_transportation', 'relevance...",0.037104,Neutral,"[{'ticker': 'CARS', 'relevance_score': '0.3334..."
3,"General Motors pivots battery strategy, puts L...",https://seekingalpha.com/news/4602466-general-...,20260611T092702,"['Manshi Mamtora', 'CFA']",General Motors is reportedly shifting its elec...,https://static.seekingalpha.com/cdn/s3/uploads...,Seeking Alpha,General,Seeking Alpha,"[{'topic': 'energy_transportation', 'relevance...",0.138295,Neutral,"[{'ticker': 'GM', 'relevance_score': '1.000000..."
4,Musk to discuss Terafab chip plant plans at AS...,https://seekingalpha.com/news/4602459-musk-to-...,20260611T081040,['Preeti Singh'],Elon Musk is scheduled to virtually attend ASM...,https://static.seekingalpha.com/cdn/s3/uploads...,Seeking Alpha,General,Seeking Alpha,"[{'topic': 'technology', 'relevance_score': '0...",0.172115,Somewhat-Bullish,"[{'ticker': 'ASML', 'relevance_score': '1.0000..."


# Convert to DataFrame

In [4]:
print("Shape:", api_raw_df.shape)
print("Columns:")
print(api_raw_df.columns)

display(api_raw_df.head())

Shape: (50, 13)
Columns:
Index(['title', 'url', 'time_published', 'authors', 'summary', 'banner_image',
       'source', 'category_within_source', 'source_domain', 'topics',
       'overall_sentiment_score', 'overall_sentiment_label',
       'ticker_sentiment'],
      dtype='object')


,title,url,time_published,authors,summary,banner_image,source,category_within_source,source_domain,topics,overall_sentiment_score,overall_sentiment_label,ticker_sentiment
0,UBS reiterates Packaging Corp. of America stoc...,https://ca.investing.com/news/stock-market-new...,20260611T143829,['Investing.com'],UBS has reiterated a Buy rating and a $248.00 ...,https://i-invdn-com.investing.com/news/LYNXMPE...,Investing.com Canada,General,Investing.com Canada,"[{'topic': 'earnings', 'relevance_score': '1.0...",0.049732,Neutral,"[{'ticker': 'PKG', 'relevance_score': '1.00000..."
1,Oppenheimer raises Tesla stock price target on...,https://www.investing.com/news/analyst-ratings...,20260611T132540,['Investing.com'],Oppenheimer has increased its price target for...,https://i-invdn-com.investing.com/news/moved_L...,Investing.com,General,Investing.com,"[{'topic': 'financial_markets', 'relevance_sco...",0.203912,Somewhat-Bullish,"[{'ticker': 'TSLA', 'relevance_score': '1.0000..."
2,Is the 2026 Toyota bZ Woodland a Good Crossove...,https://www.cars.com/articles/is-the-2026-toyo...,20260611T113909,['Jim Travers'],The 2026 Toyota bZ Woodland is an electric cro...,https://images.cars.com/cldstatic/wp-content/u...,Cars.com,General,Cars.com,"[{'topic': 'energy_transportation', 'relevance...",0.037104,Neutral,"[{'ticker': 'CARS', 'relevance_score': '0.3334..."
3,"General Motors pivots battery strategy, puts L...",https://seekingalpha.com/news/4602466-general-...,20260611T092702,"['Manshi Mamtora', 'CFA']",General Motors is reportedly shifting its elec...,https://static.seekingalpha.com/cdn/s3/uploads...,Seeking Alpha,General,Seeking Alpha,"[{'topic': 'energy_transportation', 'relevance...",0.138295,Neutral,"[{'ticker': 'GM', 'relevance_score': '1.000000..."
4,Musk to discuss Terafab chip plant plans at AS...,https://seekingalpha.com/news/4602459-musk-to-...,20260611T081040,['Preeti Singh'],Elon Musk is scheduled to virtually attend ASM...,https://static.seekingalpha.com/cdn/s3/uploads...,Seeking Alpha,General,Seeking Alpha,"[{'topic': 'technology', 'relevance_score': '0...",0.172115,Somewhat-Bullish,"[{'ticker': 'ASML', 'relevance_score': '1.0000..."


# Select Relevant Columns

In [5]:
selected_columns = [
    "title",
    "url",
    "time_published",
    "summary",
    "source",
    "overall_sentiment_score",
    "overall_sentiment_label",
    "ticker_sentiment"
]

api_cleaned_df = api_raw_df[selected_columns].copy()

def extract_tsla_field(value, field):
    """Extract TSLA-specific values from Alpha Vantage ticker_sentiment."""
    try:
        ticker_items = ast.literal_eval(value) if isinstance(value, str) else value
    except Exception:
        ticker_items = []

    if not isinstance(ticker_items, list):
        return None

    for item in ticker_items:
        if item.get("ticker") == "TSLA":
            return item.get(field)
    return None

api_cleaned_df["tsla_relevance_score"] = api_cleaned_df["ticker_sentiment"].apply(
    lambda value: extract_tsla_field(value, "relevance_score")
)
api_cleaned_df["tsla_sentiment_score"] = api_cleaned_df["ticker_sentiment"].apply(
    lambda value: extract_tsla_field(value, "ticker_sentiment_score")
)
api_cleaned_df["tsla_sentiment_label"] = api_cleaned_df["ticker_sentiment"].apply(
    lambda value: extract_tsla_field(value, "ticker_sentiment_label")
)
api_cleaned_df = api_cleaned_df.drop(columns=["ticker_sentiment"])

display(api_cleaned_df.head())

,title,url,time_published,summary,source,overall_sentiment_score,overall_sentiment_label,tsla_relevance_score,tsla_sentiment_score,tsla_sentiment_label
0,UBS reiterates Packaging Corp. of America stoc...,https://ca.investing.com/news/stock-market-new...,20260611T143829,UBS has reiterated a Buy rating and a $248.00 ...,Investing.com Canada,0.049732,Neutral,0.570210,0.321158,Somewhat-Bullish
1,Oppenheimer raises Tesla stock price target on...,https://www.investing.com/news/analyst-ratings...,20260611T132540,Oppenheimer has increased its price target for...,Investing.com,0.203912,Somewhat-Bullish,1.000000,0.463278,Bullish
2,Is the 2026 Toyota bZ Woodland a Good Crossove...,https://www.cars.com/articles/is-the-2026-toyo...,20260611T113909,The 2026 Toyota bZ Woodland is an electric cro...,Cars.com,0.037104,Neutral,0.639476,0.040831,Neutral
3,"General Motors pivots battery strategy, puts L...",https://seekingalpha.com/news/4602466-general-...,20260611T092702,General Motors is reportedly shifting its elec...,Seeking Alpha,0.138295,Neutral,0.644370,0.171347,Somewhat-Bullish
4,Musk to discuss Terafab chip plant plans at AS...,https://seekingalpha.com/news/4602459-musk-to-...,20260611T081040,Elon Musk is scheduled to virtually attend ASM...,Seeking Alpha,0.172115,Somewhat-Bullish,0.836416,0.208840,Somewhat-Bullish


# Clean Data

In [6]:
api_cleaned_df = api_cleaned_df.dropna(subset=["title", "url", "time_published", "tsla_relevance_score"])

api_cleaned_df["title"] = api_cleaned_df["title"].astype(str).str.strip()
api_cleaned_df["url"] = api_cleaned_df["url"].astype(str).str.strip()
api_cleaned_df["summary"] = api_cleaned_df["summary"].fillna("").astype(str).str.strip()
api_cleaned_df["source"] = api_cleaned_df["source"].fillna("").astype(str).str.strip()

api_cleaned_df["time_published"] = pd.to_datetime(
    api_cleaned_df["time_published"],
    format="%Y%m%dT%H%M%S",
    errors="coerce"
)
api_cleaned_df["date"] = api_cleaned_df["time_published"].dt.date

numeric_columns = ["overall_sentiment_score", "tsla_relevance_score", "tsla_sentiment_score"]
for column in numeric_columns:
    api_cleaned_df[column] = pd.to_numeric(api_cleaned_df[column], errors="coerce")

api_cleaned_df = api_cleaned_df.dropna(subset=["time_published", "date"])
api_cleaned_df = api_cleaned_df.drop_duplicates(subset=["url"])
api_cleaned_df = api_cleaned_df.sort_values("time_published", ascending=False)

print("Cleaned API rows:", len(api_cleaned_df))
display(api_cleaned_df.head())

Cleaned API rows: 50


,title,url,time_published,summary,source,overall_sentiment_score,overall_sentiment_label,tsla_relevance_score,tsla_sentiment_score,tsla_sentiment_label,date
0,UBS reiterates Packaging Corp. of America stoc...,https://ca.investing.com/news/stock-market-new...,2026-06-11 14:38:29,UBS has reiterated a Buy rating and a $248.00 ...,Investing.com Canada,0.049732,Neutral,0.570210,0.321158,Somewhat-Bullish,2026-06-11
1,Oppenheimer raises Tesla stock price target on...,https://www.investing.com/news/analyst-ratings...,2026-06-11 13:25:40,Oppenheimer has increased its price target for...,Investing.com,0.203912,Somewhat-Bullish,1.000000,0.463278,Bullish,2026-06-11
2,Is the 2026 Toyota bZ Woodland a Good Crossove...,https://www.cars.com/articles/is-the-2026-toyo...,2026-06-11 11:39:09,The 2026 Toyota bZ Woodland is an electric cro...,Cars.com,0.037104,Neutral,0.639476,0.040831,Neutral,2026-06-11
3,"General Motors pivots battery strategy, puts L...",https://seekingalpha.com/news/4602466-general-...,2026-06-11 09:27:02,General Motors is reportedly shifting its elec...,Seeking Alpha,0.138295,Neutral,0.644370,0.171347,Somewhat-Bullish,2026-06-11
4,Musk to discuss Terafab chip plant plans at AS...,https://seekingalpha.com/news/4602459-musk-to-...,2026-06-11 08:10:40,Elon Musk is scheduled to virtually attend ASM...,Seeking Alpha,0.172115,Somewhat-Bullish,0.836416,0.208840,Somewhat-Bullish,2026-06-11


# Check Missing Values

In [7]:
api_cleaned_df.isnull().sum()

title                      0
url                        0
time_published             0
summary                    0
source                     0
overall_sentiment_score    0
overall_sentiment_label    0
tsla_relevance_score       0
tsla_sentiment_score       0
tsla_sentiment_label       0
date                       0
dtype: int64

# Save Data

In [8]:
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

api_raw_df.to_csv("../data/raw/alpha_vantage_tesla_news_raw.csv", index=False)
api_cleaned_df.to_csv("../data/processed/alpha_vantage_tesla_news_cleaned.csv", index=False)

print("Saved raw data to ../data/raw/alpha_vantage_tesla_news_raw.csv")
print("Saved cleaned data to ../data/processed/alpha_vantage_tesla_news_cleaned.csv")
print("Final cleaned shape:", api_cleaned_df.shape)

Saved raw data to ../data/raw/alpha_vantage_tesla_news_raw.csv
Saved cleaned data to ../data/processed/alpha_vantage_tesla_news_cleaned.csv
Final cleaned shape: (50, 11)


# Quick Analysis

In [9]:
print("Date range:")
print(api_cleaned_df["date"].min(), "to", api_cleaned_df["date"].max())

print("\nOverall sentiment label counts:")
print(api_cleaned_df["overall_sentiment_label"].value_counts())

print("\nTSLA-specific sentiment label counts:")
print(api_cleaned_df["tsla_sentiment_label"].value_counts())

print("\nAverage overall sentiment score:")
print(api_cleaned_df["overall_sentiment_score"].mean())

print("\nAverage TSLA-specific sentiment score:")
print(api_cleaned_df["tsla_sentiment_score"].mean())


Date range:
2026-06-09 to 2026-06-11

Overall sentiment label counts:
overall_sentiment_label
Neutral             25
Somewhat-Bullish    15
Somewhat-Bearish     4
Bearish              3
Bullish              3
Name: count, dtype: int64

TSLA-specific sentiment label counts:
tsla_sentiment_label
Neutral             22
Somewhat-Bullish    15
Somewhat-Bearish     6
Bearish              4
Bullish              3
Name: count, dtype: int64

Average overall sentiment score:
0.081795

Average TSLA-specific sentiment score:
0.07749992


# Short Conclusion

This notebook collected Tesla-related financial news from the Alpha Vantage News & Sentiment API.  
The raw API response was cleaned and reduced to the most relevant columns for the project.

A key part of this step was extracting TSLA-specific relevance and sentiment values from the `ticker_sentiment` field.  
The final cleaned dataset contains 50 news articles and includes both overall article sentiment and TSLA-specific sentiment.

The cleaned API dataset is saved as `alpha_vantage_tesla_news_cleaned.csv` and is used later for merging with the stock data and scraped news data.